In [18]:
import os
import numpy as np
from netCDF4 import Dataset
import xarray as xr
from datetime import date, datetime, timezone
import cftime
import matplotlib.pyplot as plt
import pandas as pd
import warnings

In [19]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

# Thinning from LUH3 Harvested Biomass

## Define Input Arguments

In [3]:
write_files = True     # Optional flag: To choose whether the write the newly created datasets to NetCDF
save_files_path = '/g/data/p66/ajn563/ACCESS-ESM/ESM1.6/luh3-1-1/'   # Directory path for location of files generated with this script

## Define the ACCESS land cover types that we apply thinning (wood harvest) to.
##  - Typically assumed to be woody tree PFTs only
harvest_pfts = [1, 2, 3, 4, 12, 13]

In [4]:
# Check 
if write_files:
    if 'save_files_path' not in locals():
        raise ValueError("`save_files_path` must be defined when write_files == True.")
    if not os.path.exists(save_files_path):
        raise FileNotFoundError(f"The directory for saving output files '{save_files_path}' does not exist.")

## Import Relevant Datasets

#### LUH3 Harvest Biomass Data

In [5]:
f = "/g/data/p66/ajn563/ACCESS-ESM/ESM1.6/luh3-1-1/harvest_bioh.nc"

ds_accessluh3_harv = xr.open_dataset(f, decode_times=time_coder)

ds_accessluh3_harv = ds_accessluh3_harv.sel(time=slice('1850','2023')).rename({'lat': 'latitude', 'lon': 'longitude'})

#### LUH3 States Data Remapped to ACCESS Grid

In [6]:
files = "/g/data/p66/ajn563/ACCESS-ESM/ESM1.6/luh3-1-1/states_*_remap.nc"

ds_accessluh3states = xr.open_mfdataset(files, decode_times=time_coder);

ds_accessluh3states = ds_accessluh3states.rename({"lat":"latitude", "lon":"longitude"});

#### LUH3 Transitions Data Aligned and Normalised to ACCESS Grid

In [7]:
f = "/g/data/p66/ajn563/ACCESS-ESM/ESM1.6/luh3-1-1/transitions_remap_normed_1850-2023.nc"
ds_accessluh3_trans = xr.open_dataset(f, decode_times=time_coder)

#### LUH3 Static Data (Ice and Water)

In [8]:
f = "/g/data/p66/ajn563/ACCESS-ESM/ESM1.6/luh3-1-1/luh3_icwtr_remap.nc"

ds_accessluh3icwtr = xr.open_dataset(f, decode_times=time_coder)
ds_accessluh3icwtr = ds_accessluh3icwtr.rename({"lat": "latitude", "lon":"longitude"})

#### ACCESS Grid Data (land fraction and cell area)

In [9]:
f = "/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/piControl/r1i1p1f1/fx/sftlf/gn/latest/sftlf_fx_ACCESS-ESM1-5_piControl_r1i1p1f1_gn.nc"
ds_access_sftlf = xr.open_dataset(f).rename({'lat': 'latitude', 'lon': 'longitude'})

f = "/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/piControl/r1i1p1f1/fx/areacella/gn/latest/areacella_fx_ACCESS-ESM1-5_piControl_r1i1p1f1_gn.nc"
ds_access_carea = xr.open_dataset(f).rename({'lat': 'latitude', 'lon': 'longitude'})

#### ACCESS-ESM1.5 Pre-Industrial Land Cover Map

Used when normalising the LUH3 data to ensure we account for existing fractional area wetlands, lakes and permanent ice in ACCESS

In [10]:
f = "/g/data/p66/ajn563/ACCESS-ESM/ESM1.6/luh3-1-1/vegfrac_HI-05.1850.nc"

ds_accessvegfrac1850 = xr.open_dataset(f,decode_times=time_coder)

ds_accessvegfrac1850 = ds_accessvegfrac1850.rename({"lat_v": "latitude", "lon_u": "longitude"})

#### ACCESS-ESM1.6 Historical Land Cover Map

In [11]:
f = "/g/data/p66/ajn563/ACCESS-ESM/ESM1.6/luh3-1-1/ACCESS_vegfrac_LUH3_states_withAusPFTs_1850-2023_v7.nc"

ds_landcover_esm16 = xr.open_dataset(f, decode_times=time_coder)

## 1. Allocate harvested biomass to ACCESS PFTs

In [12]:
# Determine harvest fraction of tree PFTs by multiplying by the weighted tree PFT fractions
tree_pfts = ds_landcover_esm16['fraction'].sel(vegtype=harvest_pfts)
harvest_biomass = ds_accessluh3_harv['bioh']

# Calculate tree harvest fractions, using a 'safe' denominator (as sum may be 0 at some cells)
den = tree_pfts.sum('vegtype')
tree_harvest_biomass = xr.where(
    den > 0,
    harvest_biomass * tree_pfts / den,
    0
)

# Mask out ocean points
mask = ds_access_sftlf['sftlf'] > 0
tree_harvest_biomass_masked = tree_harvest_biomass.where(mask)

# Expand back to full vegtype domain, others set to 0 on land
full_veg = ds_landcover_esm16['fraction']['vegtype']   # original tile coordinate
da_expanded = tree_harvest_biomass_masked.reindex(vegtype=full_veg, fill_value=0)

# Reapply mask so the newly created vegtypes are NaN over ocean too
da_expanded = da_expanded.where(mask)

# Convert from kg to gC/m2 units
da_harv_intensity = da_expanded * 1e3 / ( ds_access_carea['areacella'] * ds_access_sftlf['sftlf']/100)

#### Modify xr.DataArray attributes for comptability with ACCESS output below

In [13]:
da_harv_intensity = da_harv_intensity.rename({'latitude':'lat', 'longitude':'lon', 'vegtype':'pseudo_level_0'})

da_harv_intensity = da_harv_intensity.assign_coords(
    time=xr.date_range(
        start=str(da_harv_intensity.time.dt.year.values[0]),
        periods=da_harv_intensity.sizes["time"],
        freq="YS",
        calendar="proleptic_gregorian",
        use_cftime=True,
    )
)

da_harv_intensity = da_harv_intensity.assign_coords(pseudo_level_0=da_harv_intensity['pseudo_level_0'].astype(int))

## 2. Calculate thinning area fraction from harvest intensity

### 2.1 Calculate forest thinning (area fraction) using harvest intensity (g C m-2) multiplied by a global scaling factor

In [14]:
harv_to_wood_scaling_factor = 0.002

In [15]:
da_forest_thin_scaling = harv_to_wood_scaling_factor * da_harv_intensity.squeeze()

da_forest_thin_tau = xr.where(
    da_forest_thin_scaling > 1e-12,
    1.0 / da_forest_thin_scaling,
    np.nan
)

### 2.2 Impose temporal cumulative constraint on thinning

This imposes a rolling cumulative constraint on the forest thinning fraction (per grid cell) to prevent unrealistically frequent biomass removal, considering harvest pressure is represented as a spatially uniform, area‑averaged thinning on forested tiles. This constraint limits cumulative thinning to at most 40% ($F_{max}$) of the standing wood carbon over a 30-year window ($t_{window}$). This formulation still allows for occasional intense thinning, provided it is offset by periods of lower activity. If thinning were applied continuously at the maximum allowed cumulative rate, the constraint would correspond to an effective rotation time of approximately 75 years (i.e. 30/0.4), which is conservative for area‑averaged forest management. This serves as a surrogate for forest recovery times that are not explicitly represented in ACCESS‑ESM1.6, nor encoded in the LUH3 harvested biomass data, while preserving the spatiotemporal patterns of harvest in LUH3. This constraint has a very small effect on global wood harvest totals (<2%) but can have a sizeable impact in certain locations where the LUH3 data implies significant, repeated wood harvest. 

In [16]:
t_window = 30   # time window for Fmax constraint (years)
Fmax = 0.4      # max allowable forest harvest fraction over t_window period (-)

f_cum = da_forest_thin_scaling.rolling(time=t_window, min_periods=1).sum()

da_forest_thin_scaling_limited = xr.where(
    f_cum > Fmax,
    da_forest_thin_scaling * (Fmax / f_cum),
    da_forest_thin_scaling
)

da_forest_thin_tau_limited = xr.where(
    da_forest_thin_scaling_limited > 1e-12,
    1.0 / da_forest_thin_scaling_limited,
    np.nan
)

### 2.4 Optional: Estimate potential ACCESS-ESM1.6 wood harvest from thinning fraction

Tests can be conducted with this section to evaluate how changes in thinning fraction input (including the choice of global scaling factor and temporal cumulative constraint) may impact global wood harvest and the historical land carbon budget, including the land-use change related emissions. To assess how changes in wood harvest impact the land carbon budget and land-use change emissions, you must use the potential wood harvest field created below ('da_forest_thin_limited_harv_PgCyr') as input the the wood products and respiration model (as used in the CABLE code). More details below.

Based on a spatiotemporal map of wood harvest intensity (forest management only i.e. wood harvest without a land-cover transition) and a simulation of wood carbon without thinning on. 

N.B. this may overestimate the actual amount of harvest possible given that we are not dynamically adjusting the wood carbon pool following each annual harvest event (as would occur inside of the model). There may also be compensating effects of the forest PFT in NEE (NPP and/or Rh) due to disequilibrium induced by the harvest event. 

To approximate the harvestable wood products from forest thinning, $HWP_{thin}$, we do the following:

$HWP_{thin} = \alpha \; I_{harv} \; C_{wood}$

where $I_{harv}$ is the wood harvest intensity from LUH3 (g C m-2, land-area basis), $C_{wood}$ is the wood carbon density (g C m-2, land-area basis) in harvestable tree PFTs from a ACCESS-ESM1.6 historical simulation without thinning switched on (concentration-driven, with land-cover change on), and $\alpha$ is a scaling factor. Therefore, $\alpha \times I_{harv}$ is the area-based forest thinning fraction. 

We conduct sensitivity tests on $\alpha$ to determine an appropriate area-based forest thinning fraction. 

#### Load historical wood carbon field from an ACCESS-ESM1.6 simulation without thinning

In [28]:
# # Example output: Annual wood carbon field 'fld_s03i853' with dimensions (time: 172, lon: 192, lat: 145, pseudo_level_0: 17)
# f = '/scratch/p66/ajn563/summary_hist-mar26-02-1c87d12c_pools/hist-mar26-02-1c87d12c_pools_fld_s03i853.ann.nc'
# ds_esmhist_nothin_cwood = xr.open_dataset(f, decode_times=time_coder)

# ds_esmhist_nothin_cwood = ds_esmhist_nothin_cwood.sel(time=slice('1850','2021'))

# # Move time index to start of year to match thinning data
# ds_esmhist_nothin_cwood = ds_esmhist_nothin_cwood.assign_coords(
#                         time=[
#                             cftime.DatetimeProlepticGregorian(
#                                 t.year, 1, 1,
#                                 has_year_zero=t.has_year_zero
#                             )
#                             for t in ds_esmhist_nothin_cwood.time.values
#                         ]
#                     )

# # ACCESS-ESM1.6 land fraction
# f = '/scratch/p66/ajn563/summary_hist-mar26-02-1c87d12c_fluxes/hist-mar26-02-1c87d12c_fluxes_fld_s03i395.ann.nc'
# ds_esmhist_nothin_landfrac = xr.open_dataset(f, decode_times=time_coder)

# # ACCESS-ESM1.6 tile fraction
# f = '/scratch/p66/ajn563/summary_hist-mar26-02-1c87d12c_fluxes/hist-mar26-02-1c87d12c_fluxes_fld_s03i317.ann.nc'
# ds_esmhist_nothin_tilefrac = xr.open_dataset(f, decode_times=time_coder)

# # - Move time index to start of year to match thinning data
# ds_esmhist_nothin_tilefrac_tyear0 = ds_esmhist_nothin_tilefrac.assign_coords(time=[cftime.DatetimeProlepticGregorian(t.year, 1, 1,has_year_zero=t.has_year_zero) for t in ds_esmhist_nothin_tilefrac.time.values])

# # ACCESS-ESM1.5 grid cell area
# f = "/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r1i1p1f1/fx/areacella/gn/latest/areacella_fx_ACCESS-ESM1-5_historical_r1i1p1f1_gn.nc"
# ds_esm15_areacella = xr.open_dataset(f, decode_times=time_coder)
# da_esm15_areacella = ds_esm15_areacella['areacella']

#### Calculate potential wood harvest from the wood carbon model output

In [103]:
# da_forest_thin_limited_harv_PgCyr = da_forest_thin_scaling_limited.sel(time=slice('1850','2021')).squeeze() * ds_esmhist_nothin_cwood['fld_s03i853'].squeeze() * ds_esmhist_nothin_landfrac['fld_s03i395'].isel(time=0).squeeze() * ds_esmhist_nothin_tilefrac_tyear0['fld_s03i317'] * da_esm15_areacella / 1e15

In [104]:
# fig, axes = plt.subplots(1,2,figsize=(12,4),sharex=True)

# x = np.arange(1850,2015)

# ax = axes[0]
# y = da_forest_thin_limited_harv_PgCyr.sum(('pseudo_level_0','lat','lon')).values
# ax.plot(np.arange(1850,2022), y, c='0.5', label='all harvest variables')

# ax.set_ylabel("Pg C yr-1")
# ax.set_title('Annual wood harvest from thinning: Offline estimate')
# ax.grid(True, alpha=0.5)

# ax = axes[1]
# ax.plot(np.arange(1850,2022), np.cumsum(y), c='0.5', label='all harvest variables')
# ax.set_ylabel("Pg C")
# ax.set_title('Cumulative wood harvest from thinning: Offline estimate')
# ax.grid(True, alpha=0.5)
# ax.legend()
# ax.set_xlim([1850,2025])

## Write to netCDF file

### LUH3 Harvest Intensity

In [44]:
ds_output = da_harv_intensity.to_dataset(name="harvest")

## Set to the expected calendar format
ds_output = ds_output.convert_calendar("proleptic_gregorian", use_cftime=True)

# CF encoding for time when writing
start_year = ds_output.time.values[0].year
end_year = int(ds_output['time'].dt.year.values[-1])
ds_output.time.encoding = {
    "units": f"days since {start_year}-01-01 00:00:00",
    "calendar": "proleptic_gregorian",
}

## Add attributes
ds_output['harvest'].attrs = {
    "standard_name": "Wood harvest biomass intensity",
    "long_name": "Wood harvest biomass carbon intensity from all vegetation in LUH3, per m2 land area (not grid cell area)",
    # "long_name": "Wood harvest biomass carbon intensity from 'primf' in LUH3, per m2 land area (not grid cell area)",
    "units": "gC/m2",
}

ds_output.attrs = {
    "title": "ACCESS-ESM1.6 Historical Forest Harvest Intesity",
    "summary": "Wood harvest biomass intensity maps for 17 ACCESS land cover classes for the years 1850-2023. Derived from the Land-Use Harmonization v3.1.1 (LUH3.1.1) in combination with the ACCESS-ESM1.6 land cover map.",
    "institution": "Commonwealth Scientific and Industrial Research Organisation (CSIRO)",
    "source": "Derived from the input4MIPs LUH3.1.1 data prepared for CMIP7",
    "history": f"Created on {date.today()} using NCI's Gadi",
    "references": "",
    "comment": "",
    "creator_name": "Alexander Norton",
    "creator_email": "alex.norton@csiro.au",
}

# Overwrite some attribute fields
ds_output.attrs['title'] = "ACCESS-ESM1.6 Historical Forest Harvest Intesity"
ds_output.attrs['summary'] = "Wood harvest biomass intensity maps for 17 ACCESS land cover classes for the years 1850-2023. Derived from the Land-Use Harmonization v3.1.1 (LUH3.1.1) in combination with the ACCESS-ESM1.6 land cover map."
ds_output.attrs['history'] = f"Created on {date.today()} using NCI's Gadi"
ds_output.attrs['creator_name'] = "Alexander Norton"
ds_output.attrs['creator_email'] = "alex.norton@csiro.au"

if write_files:
    out_path = f"{save_files_path}/LUH3_cable_bioh_harvest_intensity_{start_year}-{end_year}.nc"
    if os.path.exists(out_path):
        warnings.warn(f"File already exists and will not be overwritten: {out_path}")
    else:
        ds_output.transpose("time", "pseudo_level_0", "lat", "lon").rename({'pseudo_level_0': 'vegtype'}).to_netcdf(out_path, format="NETCDF4", unlimited_dims=["time"])
    

### ACCESS-ESM1.6 Thinning Fraction

In [20]:
# Create dataset
ds_output = (1 - da_forest_thin_scaling_limited.sel(time=slice('1850','2023'))).to_dataset(name="fraction")

# Optional but recommended: time coordinate attributes
ds_output["time"].attrs = {
    "standard_name": "time",
    "long_name": "time",
}

## Add attributes
ds_output['fraction'].attrs = {
    "AREA_OR_POINT": "Area",
    "standard_name": "thinning_ratio",
    "long_name": "Compliment of fraction of thinned grid cell area",
    "units": "1",
}

_Fmax = int(Fmax*100)
ds_output.attrs = {
    "title": "ACCESS-ESM1.6 Historical Forest Thinning",
    "summary": f"Fractional tree (wood) thinning ratio maps for 17 ACCESS land cover classes for the years 1850-2023. Derived from the Land-Use Harmonization v3.1.1 (LUH3.1.1) harvested biomass, scaled globally to convert to area, and with a rolling cumulative constraint imposed (at most {_Fmax}% of the standing wood carbon can be thinned over a {t_window}-year window).",
    "institution": "Commonwealth Scientific and Industrial Research Organisation (CSIRO)",
    "source": "Derived from the input4MIPs LUH3.1.1 data prepared for CMIP7",
    "history": f"Created on {date.today()} using NCI's Gadi",
    "references": "",
    "comment": "",
    "creator_name": "Alexander Norton",
    "creator_email": "alex.norton@csiro.au",
}

# CF encoding for time when writing
start_year = int(ds_output['time'].dt.year.values[0])
end_year = int(ds_output['time'].dt.year.values[-1])

encoding = {
    "time": {
        "units": f"days since {start_year}-01-01 00:00:00",
        "calendar": "proleptic_gregorian",
    }
}

out_path = f"{save_files_path}/ACCESS-ESM/ESM1.6/luh3-1-1/LUH3_cable_thinning_frac_from_bioh_{start_year}-{end_year}.nc"
if os.path.exists(out_path):
    warnings.warn(f"File already exists and will not be overwritten: {out_path}")
else:
    ds_output.transpose('time','pseudo_level_0','lat','lon').rename({'pseudo_level_0': 'vegtype'}).to_netcdf(out_path, format="NETCDF4", unlimited_dims=["time"], encoding=encoding)

/jobfs/166491373.gadi-pbs/ipykernel_2050724/3975496357.py:46: UserWarning: File already exists and will not be overwritten: /g/data/p66/ajn563/ACCESS-ESM/ESM1.6/luh3-1-1/LUH3_cable_thinning_frac_from_bioh_1850-2023.nc
  warnings.warn(f"File already exists and will not be overwritten: {out_path}")
